## InfoGAN with MNIST

This notebook provides an implementation of InfoGAN using the MNIST dataset [Chen et al.](https://proceedings.neurips.cc/paper/2016/file/7c9d0b1f96aebd7b5eca8c3edaa19ebb-Paper.pdf).


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os
import itertools

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.nn.utils.parametrizations import spectral_norm
from torch.optim import Adam
from torch.autograd import Variable
import torch.nn.functional as F
# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import make_grid, save_image


# tqdm
from tqdm.notebook import tqdm, trange

In [ ]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
FloatTensor = torch.cuda.FloatTensor if torch.cuda.is_available() else torch.FloatTensor
LongTensor = torch.cuda.LongTensor if torch.cuda.is_available() else torch.LongTensor

### Load and Prepare MNIST Dataset
MNIST is a built in dataset with torchvision and we load it here and prepare dataloaders for PyTorch.

In [ ]:
transform = transforms.Compose([transforms.Pad((2, 2)), transforms.ToTensor()])
mnist_trainset = MNIST(root='./data', train=True, download=True, transform=transform)

In [ ]:
train_batch_size = 256
train_loader = data.DataLoader(mnist_trainset, batch_size=train_batch_size, 
                               shuffle=True, drop_last=True, pin_memory=True)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=100, 
                       size=(1, 32, 32), 
                       nrow=10, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def to_categorical(y, num_columns, device):
    y_cat = torch.zeros(y.shape[0], num_columns, device=device)
    y_cat[range(y.shape[0]), y] = 1.0
    return y_cat

### Build the Generator class
The generator class represents the Generator network. The network has a noise vector as input and outputs images of size (3, 32, 32). Here the input is split into noise variables of dimension 62, a categorical variables of dimension 10 and 2 continuous code variables.

In [ ]:
class Generator(nn.Module):
    def __init__(self,latent_dim=62, cat_dim = 10, code_dim = 2, out_size=32, no_channels=1):
        super(Generator, self).__init__()
        input_dim = latent_dim + cat_dim + code_dim

        self.init_size = out_size // 4  # Initial size before upsampling
        self.l1 = nn.Sequential(nn.Linear(input_dim, 128 * self.init_size ** 2))

        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, no_channels, 3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, noise, labels, code):
        gen_input = torch.cat((noise, labels, code), -1)
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

### Build the Discriminator class

The discriminator class represents the Discriminator network that will be trained to determine if an image is generated or from the real data distribution. This is a standard classification network using three dense layers with Leaky ReLU activation (note the final Sigmoid is in the loss function).


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_size=32, cat_dim = 10, code_dim = 2, no_channels=1):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            spectral_norm(nn.Conv2d(no_channels, in_size, 3, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size, in_size * 2, 3, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size * 2, in_size * 4, 3, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size * 4, in_size * 8, 3, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.adversarial = nn.Sequential(nn.Linear(in_size * 32, 1))
        self.auxiliary = nn.Sequential(
            nn.Linear(in_size * 32, cat_dim), 
            nn.Softmax())
        self.code = nn.Sequential(nn.Linear(in_size * 32, code_dim))
        
    def forward(self, image):
        out = self.disc(image)
        out = out.view(out.shape[0], -1)
        adv_out = self.adversarial(out)
        aux_out = self.auxiliary(out)
        code_out = self.code(out)
        return adv_out, aux_out, code_out

### Train the Model

In [ ]:
cat_loss = nn.CrossEntropyLoss()
cont_loss = nn.MSELoss()
n_epochs = 200
latent_dim = 62
lr = 0.0002
ilr = 0.0002
beta_1 = 0.5
beta_2 = 0.999
epoch_show = 5
show_batch = 100
n_critic = 1
n_classes = 10
lam_cat = 1
lam_cont = 0.1
cat_dim = 10
code_dim = 2

static_z = torch.zeros(cat_dim ** 2, latent_dim, device=device)
static_label = to_categorical(
    np.array([num for _ in range(cat_dim) for num in range(cat_dim)]), 
    num_columns=cat_dim,
    device=device
)
static_code = torch.zeros(cat_dim ** 2, code_dim, device=device)

In [ ]:
gen = Generator(latent_dim=latent_dim).to(device)
gen_opt = Adam(gen.parameters(), lr=lr, betas=(beta_1, beta_2))
disc = Discriminator().to(device) 
disc_opt = Adam(disc.parameters(), lr=lr, betas=(beta_1, beta_2))
info_opt = Adam(itertools.chain(gen.parameters(), disc.parameters()), lr=ilr, betas=(beta_1, beta_2))

In [ ]:
gen_loss_lst = list()
disc_loss_lst = list()
info_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_gen_loss = 0.0
    avg_disc_loss = 0.0
    avg_info_loss = 0.0
    num_items = 0
    for real_imgs, _ in tqdm(train_loader):

        cur_batch_size = real_imgs.shape[0]
        real_imgs = real_imgs.to(device)

        valid = torch.ones(cur_batch_size, 1, device=device, requires_grad=False)
        fake = torch.zeros(cur_batch_size, 1, device=device, requires_grad=False)

        #  Train Generator
        gen_opt.zero_grad()
        z = torch.randn(cur_batch_size, latent_dim, device=device)
        label_input = to_categorical(np.random.randint(0, cat_dim, cur_batch_size), 
                                     num_columns=cat_dim,
                                     device=device)
        code_input = torch.rand(cur_batch_size, code_dim, device=device) * 2 - 1.0
        gen_imgs = gen(z, label_input, code_input)
        validity, _, _ = disc(gen_imgs)
        gloss = -validity.mean()
        
        gloss.backward()
        gen_opt.step()

        #  Train Discriminator
        disc_opt.zero_grad()
        real_pred, _, _ = disc(real_imgs)
        fake_pred, _, _ = disc(gen_imgs.detach())
        dloss = fake_pred.mean() - real_pred.mean()
        dloss.backward()
        disc_opt.step()

        # Information Loss
        info_opt.zero_grad()
        sampled_labels = torch.randint(cat_dim, (cur_batch_size,), device=device)
        z = torch.randn(cur_batch_size, latent_dim, device=device)
        label_input = to_categorical(sampled_labels, num_columns=cat_dim,
                                    device=device)
        code_input = torch.rand(cur_batch_size, code_dim, device=device) * 2 - 1.0
        gen_imgs = gen(z, label_input, code_input)
        _, pred_label, pred_code = disc(gen_imgs)
        iloss = lam_cat * cat_loss(pred_label, sampled_labels) + lam_cont * cont_loss(
            pred_code, code_input
        )
        iloss.backward()
        info_opt.step()

        avg_gen_loss += gloss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        avg_info_loss += iloss.item() * cur_batch_size
        num_items += cur_batch_size
        
    if (epoch + 1) % epoch_show == 0:
        z = torch.randn(10 ** 2, latent_dim, device=device)
        static_sample = gen(z, static_label, static_code)
        
        show_tensor_images(static_sample, filename='InfoGANMNISTGrid_' + str(epoch) + '.png')
        show_tensor_images(real_imgs)
        
    ave_gen_loss = avg_gen_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    ave_info_loss = avg_info_loss / num_items
    gen_loss_lst.append(ave_gen_loss)
    disc_loss_lst.append(ave_disc_loss)
    info_loss_lst.append(ave_info_loss)
    tqdm_epoch.set_description('Ave Gen Loss: {:5f}, \
                               Ave Disc Loss: {:5f}, \
                               Ave Info Loss: {:5f}'.format(ave_gen_loss, ave_disc_loss, ave_info_loss))
    torch.save(gen.state_dict(), 'infoganmnist_genckpt.pth')
    torch.save(disc.state_dict(), 'infoganmnist_discckpt.pth')


### Plot Losses

In [ ]:
epochs = range(1, n_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, gen_loss_lst, label='Generator Loss')
ax.plot(epochs, disc_loss_lst, label='Discriminator Loss')
ax.plot(epochs, info_loss_lst, color="k", linestyle=':', label='Information Loss')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('InfoGANMNISTLosses.png', dpi=300, bbox_inches='tight')

In [ ]:
z = Variable(FloatTensor(np.random.normal(0, 1, (10 ** 2, latent_dim))))
static_sample = gen(z, static_label, static_code)
plt.figure(figsize=(12,12))
show_tensor_images(static_sample, filename='InfoGANMNISTGrid.png')